In [1]:
import json
import os
import re
from sentence_transformers import SentenceTransformer
import numpy as np
import chromadb
import hashlib

BASE_PATH = "../data/processed"

c:\Users\Playdata\Desktop\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# [제공 코드] 임베딩 모델 준비 — 지난 시간에 쓴 한국어 문장 임베딩 모델입니다(불러오는 데 잠시 걸립니다)
embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 (768차원)')

test_texts = [
    "스타 미니콘의 획득확률은 7.3539%입니다.",
    "생명의 보스 반지 상자에서 리스트레인트 링을 획득할 수 있습니다."
]

test_embeddings = embed_model.encode(
    test_texts,
    normalize_embeddings=True
)

print(test_embeddings.shape)

assert test_embeddings.shape[0] == 2
assert test_embeddings.ndim == 2
assert not np.isnan(test_embeddings).any()

print("embedding test 통과")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6124.28it/s]


임베딩 모델 준비 완료 (768차원)
(2, 768)
embedding test 통과


In [3]:
import chromadb

client = chromadb.PersistentClient(path="../data/chroma_db")

collection = client.get_or_create_collection(
    name="maple_probability", metadata={"hnsw:space": "cosine"}
)

print(collection.name)

maple_probability


In [4]:
with open("../data/processed/maple_items_process.json", 'r', encoding='utf-8') as f:
    original_data = json.load(f)

print(len(original_data))

49


In [5]:
def table_to_text(item, table):
    rows = table.get("rows")

    if not rows:
        return ""

    headers = rows[0]
    data_rows = rows[1:]

    lines = []

    lines.append(f"아이템:{item.get('name', '')}")

    section_title = table.get("section_title", "")
    if section_title:
        lines.append(f"섹션: {section_title}")

    lines.append("")

    for row in data_rows:
        row_parts = []

        for header, value in zip(headers, row):
            if value:
                row_parts.append(f"{header}: {value}")

        if row_parts:
            lines.append(" | ".join(row_parts))

    table_description = table.get("table_description", "")

    if table_description:
        lines.append("")
        lines.append("표 설명:")
        lines.append(table_description)

    return "\n".join(lines)
    

In [6]:
for item in original_data:
    for table in item['tables']:
        embedding_text = table_to_text(item, table)

        print(embedding_text)

아이템:로얄스타일
섹션: 로얄스타일

구분: 로얄스타일 | 아이템명: [스페셜 라벨] 샤이닝 블루 반다나 (남자만 획득가능) / 차밍 블루 리본 (여자만 획득가능) | 획득확률: 3.0%
구분: 로얄스타일 | 아이템명: [스페셜 라벨] 치어 스타 보이 (남) / 치어 스타 걸 (여) | 획득확률: 3.0%
구분: 로얄스타일 | 아이템명: [스페셜 라벨] 블루 트레일 (남) / 블루 트레이스 (여) | 획득확률: 3.0%
구분: 로얄스타일 | 아이템명: [스페셜 라벨] 치어링 스타 | 획득확률: 3.0%
구분: 로얄스타일 | 아이템명: [스페셜 라벨] 치어 스피릿 | 획득확률: 3.0%
구분: 로얄스타일 | 아이템명: 트윙클 블루 리본 | 획득확률: 1.5%
구분: 로얄스타일 | 아이템명: 더키 바캉스 썬캡 | 획득확률: 4.0%
구분: 로얄스타일 | 아이템명: 더키 바캉스 비치 웨어 (남) / 더키 바캉스 스윔 웨어 (여) | 획득확률: 3.0%
구분: 로얄스타일 | 아이템명: 더키 바캉스 샌들 | 획득확률: 5.0%
구분: 로얄스타일 | 아이템명: 블루 웨이브 서핑 보드 (남자만 획득가능) / 리틀 더키 프렌즈 (여자만 획득가능) | 획득확률: 3.0%
구분: 로얄스타일 | 아이템명: 바캉스 젤리 백 | 획득확률: 2.0%
구분: 로얄스타일 | 아이템명: 개구디노 모자 | 획득확률: 3.8%
구분: 로얄스타일 | 아이템명: 개구디노 한벌옷 | 획득확률: 3.5%
구분: 로얄스타일 | 아이템명: 개구디노 신발 | 획득확률: 4.0%
구분: 로얄스타일 | 아이템명: 개구디노 장갑 | 획득확률: 4.0%
구분: 로얄스타일 | 아이템명: 개구디노 | 획득확률: 4.0%
구분: 로얄스타일 | 아이템명: 흑백영화 모자 | 획득확률: 4.0%
구분: 로얄스타일 | 아이템명: 흑백영화 주인공 | 획득확률: 5.0%
구분: 로얄스타일 | 아이템명: 흑백영화 구두 | 획득확률: 5.0%
구분: 로얄스타일 | 아이템명: 흑백영화 | 획득확률: 5.0%
구분: 로얄스타일 | 아이

In [7]:
def row_to_text(item, table, headers, row):

    lines = [
        f"상품명: {item.get('name', '')}",
        "문서 유형: 표 행",
    ]

    section_title = table.get("section_title", "")

    if section_title:
        lines.append(f"섹션: {section_title}")

    lines.append("")

    row_parts = []

    for header, value in zip(headers, row):

        if value not in (None, ""):
            row_parts.append(
                f"{header}: {value}"
            )

    lines.append(
        " | ".join(row_parts)
    )

    return "\n".join(lines).strip()

In [8]:
def table_to_chunks_v2(item, table):

    rows = table.get("rows", [])

    headers = (
        rows[0]
        if rows
        else []
    )

    data_rows = (
        rows[1:]
        if len(rows) > 1
        else []
    )

    section_title = (
        table.get("section_title", "")
        or ""
    )

    table_index = table.get(
        "table_index"
    )

    chunks = []

    # --------------------------------
    # 1. row 하나당 chunk 하나
    # --------------------------------

    for row_index, row in enumerate(
        data_rows,
        start=1,
    ):

        text = row_to_text(
            item,
            table,
            headers,
            row,
        )

        if not text:
            continue

        chunks.append({
            "text": text,

            "metadata": {
                "name": item.get(
                    "name",
                    ""
                ),
                "url": item.get(
                    "url",
                    ""
                ),
                "table_index": table_index,
                "section_title": section_title,
                "chunk_type": "row",
                "row_index": row_index,
            },
        })

    # --------------------------------
    # 2. table_description 별도 chunk
    # --------------------------------

    description = (
        table.get(
            "table_description",
            ""
        )
        or ""
    ).strip()

    if description:

        description_lines = [
            f"상품명: {item.get('name', '')}",
            "문서 유형: 표 설명",
        ]

        if section_title:
            description_lines.append(
                f"섹션: {section_title}"
            )

        description_lines.extend([
            "",
            "표 설명:",
            description,
        ])

        chunks.append({
            "text": "\n".join(
                description_lines
            ),

            "metadata": {
                "name": item.get(
                    "name",
                    ""
                ),
                "url": item.get(
                    "url",
                    ""
                ),
                "table_index": table_index,
                "section_title": section_title,
                "chunk_type": "description",

                # description은 row가 아니므로
                "row_index": -1,
            },
        })

    return chunks

In [9]:
def build_all_chunks_v2(data):

    all_chunks = []

    for item in data:

        for table in item.get(
            "tables",
            []
        ):

            chunks = table_to_chunks_v2(
                item,
                table,
            )

            all_chunks.extend(chunks)

    return all_chunks

In [10]:
all_chunks_v2 = build_all_chunks_v2(
    original_data
)

print(
    "전체 chunk:",
    len(all_chunks_v2)
)

전체 chunk: 2215


In [11]:
from collections import Counter

chunk_type_count = Counter(
    chunk["metadata"]["chunk_type"]
    for chunk in all_chunks_v2
)

print(chunk_type_count)

Counter({'row': 2205, 'description': 10})


In [12]:
for chunk in all_chunks_v2[:5]:

    print("=" * 80)

    print(chunk["metadata"])

    print()

    print(chunk["text"])

{'name': '로얄스타일', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/RoyalStyle', 'table_index': 1, 'section_title': '로얄스타일', 'chunk_type': 'row', 'row_index': 1}

상품명: 로얄스타일
문서 유형: 표 행
섹션: 로얄스타일

구분: 로얄스타일 | 아이템명: [스페셜 라벨] 샤이닝 블루 반다나 (남자만 획득가능) / 차밍 블루 리본 (여자만 획득가능) | 획득확률: 3.0%
{'name': '로얄스타일', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/RoyalStyle', 'table_index': 1, 'section_title': '로얄스타일', 'chunk_type': 'row', 'row_index': 2}

상품명: 로얄스타일
문서 유형: 표 행
섹션: 로얄스타일

구분: 로얄스타일 | 아이템명: [스페셜 라벨] 치어 스타 보이 (남) / 치어 스타 걸 (여) | 획득확률: 3.0%
{'name': '로얄스타일', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/RoyalStyle', 'table_index': 1, 'section_title': '로얄스타일', 'chunk_type': 'row', 'row_index': 3}

상품명: 로얄스타일
문서 유형: 표 행
섹션: 로얄스타일

구분: 로얄스타일 | 아이템명: [스페셜 라벨] 블루 트레일 (남) / 블루 트레이스 (여) | 획득확률: 3.0%
{'name': '로얄스타일', 'url': 'https://maplestory.nexon.com/Guide/CashShop/Probability/RoyalStyle', 'table_index': 1, 'section_title': '로얄스타일', 'chunk_type': '

In [13]:
def clean_metadata(metadata):

    cleaned = {}

    for key, value in metadata.items():

        if value is None:
            continue

        if isinstance(
            value,
            (str, int, float, bool)
        ):
            cleaned[key] = value

        else:
            cleaned[key] = str(value)

    return cleaned

In [14]:
def make_chunk_id(chunk):

    metadata = chunk["metadata"]
    text = chunk["text"]

    text_hash = hashlib.sha1(
        text.encode("utf-8")
    ).hexdigest()[:12]

    return (
        f"{metadata.get('name', 'unknown')}_"
        f"{metadata.get('table_index', 'na')}_"
        f"{metadata.get('row_start', 'na')}_"
        f"{metadata.get('row_end', 'na')}_"
        f"{text_hash}"
    )

In [15]:
def save_chunks_to_chroma(
    chunks,
    collection,
    embed_model,
    batch_size=32,
):

    for start in range(
        0,
        len(chunks),
        batch_size
    ):

        batch = chunks[
            start:start + batch_size
        ]

        texts = [
            chunk["text"]
            for chunk in batch
        ]

        metadatas = [
            clean_metadata(
                chunk["metadata"]
            )
            for chunk in batch
        ]

        ids = [
            make_chunk_id(chunk)
            for chunk in batch
        ]

        # embedding
        embeddings = embed_model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        # numpy -> list
        embeddings = embeddings.tolist()

        # 재실행 가능하도록 add보다 upsert 사용
        collection.upsert(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas,
        )

    print(
        f"{len(chunks)} chunks 저장 완료"
    )

In [16]:
def search_chunks(
    query,
    collection,
    embed_model,
    n_results=5,
):

    query_embedding = embed_model.encode([query], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=[
            "documents",
            "metadatas",
            "distances",
        ],
    )

    return results

In [17]:
results = search_chunks(
    query="생명의 보스 반지 상자에서 리스트레인트 링 획득 확률은?",
    collection=collection,
    embed_model=embed_model,
    n_results=5,
)

for i in range(len(results["ids"][0])):

    print("=" * 80)

    print("distance:", results["distances"][0][i])

    print("metadata:", results["metadatas"][0][i])

    print()

    print(results["documents"][0][i])

    print()

distance: 0.2557429075241089
metadata: {'url': 'https://maplestory.nexon.com/Guide/OtherProbability/bossRingBox/ringBoxLifeJade', 'table_index': 2, 'row_end': 10, 'row_start': 1, 'name': '생명의 보스 반지 상자', 'section_title': '생명의 보스 반지 상자 아이템 획득 확률'}

상품명: 생명의 보스 반지 상자
섹션: 생명의 보스 반지 상자 아이템 획득 확률

아이템 명: 리스트레인트 링 | 획득확률: 14.51613%
아이템 명: 컨티뉴어스 링 | 획득확률: 14.51613%
아이템 명: 웨폰퍼프 - S링 | 획득확률: 8.06452%
아이템 명: 웨폰퍼프 - I링 | 획득확률: 8.06452%
아이템 명: 웨폰퍼프 - L링 | 획득확률: 8.06452%
아이템 명: 웨폰퍼프 - D링 | 획득확률: 8.06452%
아이템 명: 리스크테이커 링 | 획득확률: 8.06452%
아이템 명: 링 오브 썸 | 획득확률: 8.06452%
아이템 명: 크리데미지 링 | 획득확률: 8.06452%
아이템 명: 생명의 연마석 | 획득확률: 14.51613%

distance: 0.2812650203704834
metadata: {'section_title': '스킬 반지의 레벨 설정 확률', 'table_index': 1, 'name': '생명의 보스 반지 상자', 'row_start': 1, 'url': 'https://maplestory.nexon.com/Guide/OtherProbability/bossRingBox/ringBoxLifeJade', 'row_end': 2}

상품명: 생명의 보스 반지 상자
섹션: 스킬 반지의 레벨 설정 확률

반지 레벨: 3 | 확률: 30%
반지 레벨: 4 | 확률: 70%

distance: 0.290347695350647
metadata: {'name': '백옥의 보스 반지

In [18]:
test_queries = [
    "스타 미니콘 획득 확률은?",
    "마스터피스 레드의 A와 B는 무슨 뜻이야?",
    "생명의 보스 반지 상자 리스트레인트 링 확률",
    "위대한 진 힐라의 소울 옵션은?",
]


for query in test_queries:

    print("\n")
    print("#" * 100)
    print("QUERY:", query)
    print("#" * 100)

    results = search_chunks(
        query,
        collection,
        embed_model,
        n_results=3,
    )

    for rank in range(len(results["ids"][0])):

        metadata = results["metadatas"][0][rank]

        distance = results["distances"][0][rank]

        document = results["documents"][0][rank]

        print(f"\n[{rank + 1}]")

        print("distance:", round(distance, 4))

        print("name:", metadata.get("name"))

        print("section:", metadata.get("section_title"))

        print(document[:500])



####################################################################################################
QUERY: 스타 미니콘 획득 확률은?
####################################################################################################

[1]
distance: 0.5636
name: 미라클 서큘레이터
section: 어빌리티 등급 설정 확률
상품명: 미라클 서큘레이터
섹션: 어빌리티 등급 설정 확률

구분: 에픽→유니크 | 확률: 30.00%
구분: 유니크→레전드리 | 확률: 10.00%

[2]
distance: 0.5826
name: 미라클 서큘레이터
section: 어빌리티 옵션 등급 설정 확률
상품명: 미라클 서큘레이터
섹션: 어빌리티 옵션 등급 설정 확률

구분: 첫 번째 옵션 | 에픽 등급: 에픽 | 에픽 등급: 100.00% | 유니크 등급: 유니크 | 유니크 등급: 100.00% | 레전드리 등급: 레전드리 | 레전드리 등급: 100.00%
구분: 두 번째 옵션 및 세 번째 옵션 | 에픽 등급: 레어 | 에픽 등급: 100.00% | 유니크 등급: 에픽 | 유니크 등급: 30.00% | 레전드리 등급: 유니크 | 레전드리 등급: 15.00%
구분: 두 번째 옵션 및 세 번째 옵션 | 유니크 등급: 레어 | 유니크 등급: 70.00% | 레전드리 등급: 에픽 | 레전드리 등급: 85.00%

[3]
distance: 0.5877
name: 골드 애플
section: 골드 애플
상품명: 골드 애플
섹션: 골드 애플

구분: 골드 애플 | 아이템명: 레드 파이렛 마이스터 심볼 | 획득확률: 0.43353%
구분: 골드 애플 | 아이템명: 레드 시프 마이스터 심볼 | 획득확률: 0.43353%
구분: 골드 애플 | 아이템명: 레드 매지션 마이스터 심볼 | 획득확률: 0.43353%
구분

In [19]:
COLLECTION_NAME = (
    "maple_probability_row_v1"
)

try:
    client.delete_collection(
        COLLECTION_NAME
    )
except Exception:
    pass

collection_v2 = (
    client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={
            "hnsw:space": "cosine"
        },
    )
)

In [20]:
import hashlib


def make_chunk_id_v2(chunk):

    metadata = chunk["metadata"]

    text_hash = hashlib.sha1(
        chunk["text"].encode("utf-8")
    ).hexdigest()[:12]

    return (
        f"{metadata.get('name', 'unknown')}_"
        f"{metadata.get('table_index', 'na')}_"
        f"{metadata.get('chunk_type', 'unknown')}_"
        f"{metadata.get('row_index', 'na')}_"
        f"{text_hash}"
    )

In [21]:
def save_chunks_to_chroma_v2(
    chunks,
    collection,
    embed_model,
    batch_size=64,
):

    for start in range(
        0,
        len(chunks),
        batch_size,
    ):

        batch = chunks[
            start:start + batch_size
        ]

        texts = [
            chunk["text"]
            for chunk in batch
        ]

        metadatas = [
            clean_metadata(
                chunk["metadata"]
            )
            for chunk in batch
        ]

        ids = [
            make_chunk_id_v2(chunk)
            for chunk in batch
        ]

        embeddings = embed_model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        collection.upsert(
            ids=ids,
            documents=texts,
            embeddings=embeddings.tolist(),
            metadatas=metadatas,
        )

    print(
        f"{len(chunks)} chunks 저장 완료"
    )

In [22]:
save_chunks_to_chroma_v2(
    all_chunks_v2,
    collection_v2,
    embed_model,
)

print(
    "Chroma 저장 개수:",
    collection_v2.count()
)

2215 chunks 저장 완료
Chroma 저장 개수: 2215


In [23]:
results = search_chunks(
    query="스타 미니콘 획득 확률은?",
    collection=collection_v2,
    embed_model=embed_model,
    n_results=5,
)

In [24]:
test_queries = [
    "스타 미니콘 획득 확률은?",
    "마스터피스 레드의 A와 B는 무슨 뜻이야?",
    "생명의 보스 반지 상자 리스트레인트 링 확률",
    "위대한 진 힐라의 소울 옵션은?",
]


for query in test_queries:

    print("\n")
    print("#" * 100)
    print("QUERY:", query)
    print("#" * 100)

    results = search_chunks(
        query=query,
        collection=collection_v2,
        embed_model=embed_model,
        n_results=5,
    )

    for rank in range(len(results["ids"][0])):

        metadata = results["metadatas"][0][rank]

        distance = results["distances"][0][rank]

        document = results["documents"][0][rank]

        print(f"\n[{rank + 1}] " f"distance={distance:.4f}")

        print("name:", metadata.get("name"))

        print("type:", metadata.get("chunk_type"))

        print(document[:700])



####################################################################################################
QUERY: 스타 미니콘 획득 확률은?
####################################################################################################

[1] distance=0.5334
name: 마스터피스 레드
type: row
상품명: 마스터피스 레드
문서 유형: 표 행
섹션: 마스터피스 레드

구분: 레드라벨 모자 | 아이템명: 스타 미니콘 (A) / 미니 베어 핀 (B) | 획득확률: 7.3539%

[2] distance=0.5374
name: 마스터피스 블랙
type: row
상품명: 마스터피스 블랙
문서 유형: 표 행
섹션: 마스터피스 블랙

구분: 블랙라벨 모자 | 아이템명: 스타 미니콘 (A) / 미니 베어 핀 (B) | 획득확률: 5.8135%

[3] distance=0.5846
name: 미라클 서큘레이터
type: row
상품명: 미라클 서큘레이터
문서 유형: 표 행
섹션: 어빌리티 등급 설정 확률

구분: 유니크→레전드리 | 확률: 10.00%

[4] distance=0.5851
name: 미라클 서큘레이터
type: row
상품명: 미라클 서큘레이터
문서 유형: 표 행
섹션: 어빌리티 옵션 등급 설정 확률

구분: 첫 번째 옵션 | 에픽 등급: 에픽 | 에픽 등급: 100.00% | 유니크 등급: 유니크 | 유니크 등급: 100.00% | 레전드리 등급: 레전드리 | 레전드리 등급: 100.00%

[5] distance=0.5856
name: 골드 애플
type: row
상품명: 골드 애플
문서 유형: 표 행
섹션: 골드 애플

구분: 골드 애플 | 아이템명: 150제 스타포스 17성 100% 강화권 | 획득확률: 0.001703%


########################